In [ ]:
from ultralytics import YOLO
import os
from IPython.display import Image, display
import cv2
import matplotlib.pyplot as plt

In [ ]:
model = YOLO('yolov5s.pt')

results = model.train(
    data='dataset/HRPlanes_v8/data.yaml',
    epochs=1,
    imgsz=640,
    batch=32,
    optimizer='SGD',
    cos_lr=True,
    device=0,
    workers=16,
    patience=20, # остановка
    plots=True,
    project='./runs',
    name='exp1',
    exist_ok=True,
    val=True,
    mosaic=0.8,
    mixup=0.1,
    copy_paste=0.1
)

In [ ]:
run_dir = './runs/exp1'

plots = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']
for plot in plots:
    path = os.path.join(run_dir, plot)
    if os.path.exists(path):
        print(f"\n{plot}:")
        display(Image(path))
    else:
        print(f"{plot} не найден (возможно, обучение не завершилось или plots=False)")

In [ ]:
best_model_path = os.path.join(run_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)
metrics = best_model.val(split='test', plots=True)

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision:{metrics.box.mp:.4f}")
print(f"Recall:   {metrics.box.mr:.4f}")

In [ ]:
imgs = ['dataset/HRPlanes_v8/images/val/SIN_069.jpg', 'dataset/HRPlanes_v8/images/val/LAS_0029.jpg']

results = model.predict(imgs, conf=0.25, iou=0.45)

for res, path in zip(results, imgs):
    img_bgr = res.plot()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6,6))
    plt.imshow(img_rgb)
    plt.title(os.path.basename(path))
    plt.axis('off')
    plt.show()